# Pharma Pricing Copilot — Exploratory Data Analysis

This notebook walks through the synthetic pharmaceutical pricing dataset generated by the
`PricingDataSimulator`, applies the anomaly detection engine, and visualises key insights
that inform the design of the Streamlit dashboard.

## Sections
1. Environment Setup
2. Data Generation
3. Drug Catalog EDA
4. Price History EDA
5. Gross-to-Net Waterfall Analysis
6. Anomaly Detection
7. Competitive Landscape
8. Key Findings Summary

## 1. Environment Setup

In [ ]:
"""
Environment Setup
-----------------
Ensure the project root is on sys.path so local src imports work
regardless of where the notebook kernel is launched from.
"""
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

print('Environment ready.')

## 2. Data Generation

In [ ]:
"""
Data Generation
---------------
Use the PricingDataSimulator to create a realistic synthetic dataset.
The same seed is used throughout so results are fully reproducible.
"""
from src.data_simulator import PricingDataSimulator

SEED = 42
N_DRUGS = 25
N_PERIODS = 36  # 3 years of monthly data
ANOMALY_RATE = 0.04

sim = PricingDataSimulator(seed=SEED)

catalog = sim.generate_drug_catalog(n_drugs=N_DRUGS)
price_history = sim.generate_price_history(catalog, periods=N_PERIODS)
price_history_with_anomalies = sim.inject_anomalies(price_history, anomaly_rate=ANOMALY_RATE)
competitive = sim.generate_competitive_landscape(catalog)

print(f'Catalog shape:       {catalog.shape}')
print(f'Price history shape: {price_history_with_anomalies.shape}')
print(f'Competitive shape:   {competitive.shape}')

## 3. Drug Catalog EDA

In [ ]:
"""
Drug Catalog EDA
----------------
Inspect the drug catalog: distribution by therapeutic area,
dosage form mix, and WAC distribution at launch.
"""
print('=== Drug Catalog — First 5 Rows ===')
display(catalog.head())

print('\n=== Descriptive Statistics ===')
display(catalog[['base_wac', 'annual_price_increase_pct', 'strength_mg']].describe())

In [ ]:
"""
Therapeutic Area & Dosage Form Distribution
"""
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Therapeutic area
ta_counts = catalog['therapeutic_area'].value_counts()
axes[0].barh(ta_counts.index, ta_counts.values, color=sns.color_palette('muted', len(ta_counts)))
axes[0].set_title('Drugs by Therapeutic Area')
axes[0].set_xlabel('Count')

# Dosage form
df_counts = catalog['dosage_form'].value_counts()
axes[1].pie(df_counts.values, labels=df_counts.index, autopct='%1.0f%%', startangle=90)
axes[1].set_title('Dosage Form Mix')

plt.tight_layout()
plt.show()

In [ ]:
"""
Base WAC Distribution (log scale)
"""
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(catalog['base_wac'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xscale('log')
ax.set_xlabel('Launch WAC (USD, log scale)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Launch WAC Prices')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 4. Price History EDA

In [ ]:
"""
Price History Overview
"""
ph = price_history_with_anomalies.copy()
ph['period'] = pd.to_datetime(ph['period'])

print('=== Price History — First 5 Rows ===')
display(ph.head())

print('\n=== Null Value Counts ===')
print(ph.isnull().sum())

In [ ]:
"""
WAC Trends — Top 5 Drugs
"""
top5 = ph.groupby('ndc')['wac_per_unit'].max().nlargest(5).index
trend_df = ph[ph['ndc'].isin(top5)]

fig = px.line(
    trend_df, x='period', y='wac_per_unit', color='brand_name',
    title='Monthly WAC — Top 5 Drugs by Peak Price',
    labels={'wac_per_unit': 'WAC (USD)', 'period': 'Period', 'brand_name': 'Drug'},
    template='plotly_white'
)
fig.show()

In [ ]:
"""
Price Corridor — WAC vs ASP vs AMP vs Net Price
for a single representative drug
"""
sample_ndc = catalog.iloc[0]['ndc']
sample_name = catalog.iloc[0]['brand_name']
sample_df = ph[ph['ndc'] == sample_ndc].sort_values('period')

fig = go.Figure()
for col, colour, name in [
    ('wac_per_unit', '#1F77B4', 'WAC'),
    ('asp',          '#FF7F0E', 'ASP'),
    ('amp',          '#2CA02C', 'AMP'),
    ('net_price',    '#D62728', 'Net Price'),
    ('ceiling_340b', '#9467BD', '340B Ceiling'),
]:
    if col in sample_df.columns:
        fig.add_trace(go.Scatter(
            x=sample_df['period'], y=sample_df[col],
            mode='lines', name=name, line=dict(color=colour)
        ))

fig.update_layout(
    title=f'Price Corridor — {sample_name}',
    xaxis_title='Period', yaxis_title='Price (USD)',
    template='plotly_white'
)
fig.show()

## 5. Gross-to-Net Waterfall Analysis

In [ ]:
"""
Gross-to-Net Distribution
"""
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(
    ph['gross_to_net_pct'].dropna() * 100,
    bins=30, color='darkorange', edgecolor='white', alpha=0.85
)
ax.set_xlabel('Gross-to-Net Discount (%)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Gross-to-Net Percentages Across All Drugs & Periods')
plt.tight_layout()
plt.show()

print(f"Mean GTN:   {ph['gross_to_net_pct'].mean()*100:.1f}%")
print(f"Median GTN: {ph['gross_to_net_pct'].median()*100:.1f}%")

In [ ]:
"""
GTN Heatmap — Therapeutic Area × Quarter
"""
merged = ph.merge(catalog[['ndc', 'therapeutic_area']], on='ndc', how='left')
merged['quarter'] = pd.to_datetime(merged['period']).dt.to_period('Q').astype(str)
heatmap = merged.groupby(['therapeutic_area', 'quarter'])['gross_to_net_pct'].mean().unstack()

plt.figure(figsize=(14, 6))
sns.heatmap(
    heatmap * 100,
    annot=True, fmt='.1f',
    cmap='RdYlGn_r',
    linewidths=0.5,
    cbar_kws={'label': 'GTN Discount (%)'}
)
plt.title('Average Gross-to-Net (%) by Therapeutic Area and Quarter')
plt.tight_layout()
plt.show()

## 6. Anomaly Detection

In [ ]:
"""
Run Anomaly Detection Engine
"""
from src.anomaly_detection import AnomalyDetector

detector = AnomalyDetector(
    z_threshold=3.0,
    mom_threshold=0.15,
    use_isolation_forest=True,
    use_lof=True,
    run_regulatory_checks=True,
)

detected = detector.detect(ph)
summary = detector.summary(detected)

print('=== Anomaly Detection Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
"""
Anomaly Visualisation — WAC scatter
"""
det_vis = detected.copy()
det_vis['Status'] = det_vis['anomaly'].map({True: 'Anomaly', False: 'Normal'})

fig = px.scatter(
    det_vis, x='period', y='wac_per_unit',
    color='Status',
    color_discrete_map={'Anomaly': '#EF553B', 'Normal': '#636EFA'},
    hover_data=['brand_name', 'anomaly_reason', 'anomaly_score'],
    title='WAC Prices — Anomalies Highlighted',
    labels={'wac_per_unit': 'WAC (USD)', 'period': 'Period'},
    template='plotly_white',
    opacity=0.7,
)
fig.show()

In [ ]:
"""
Anomaly Reason Breakdown
"""
flagged = detected[detected['anomaly']]
reason_counts = flagged['anomaly_reason'].value_counts()

fig = px.bar(
    reason_counts.reset_index(),
    x='anomaly_reason', y='count',
    title='Anomaly Count by Detection Method',
    labels={'anomaly_reason': 'Reason', 'count': 'Count'},
    template='plotly_white',
    color='count',
    color_continuous_scale='Reds'
)
fig.show()

print('\n=== Top 10 Highest-Score Anomalies ===')
display(
    flagged[['brand_name', 'period', 'wac_per_unit', 'anomaly_reason', 'anomaly_score']]
    .sort_values('anomaly_score', ascending=False)
    .head(10)
)

## 7. Competitive Landscape

In [ ]:
"""
Competitive Pricing Overview
"""
display(competitive.head(10))

fig = px.box(
    competitive,
    x='therapeutic_area', y='competitor_wac',
    title='Competitor WAC Distribution by Therapeutic Area',
    labels={'competitor_wac': 'Competitor WAC (USD)', 'therapeutic_area': 'Area'},
    template='plotly_white',
    color='therapeutic_area'
)
fig.update_xaxes(tickangle=30)
fig.show()

## 8. Key Findings Summary

In [ ]:
"""
Key Findings
------------
Programmatic summary of the most important EDA insights.
"""
n_total = len(detected)
n_anomalies = detected['anomaly'].sum()
top_reason = detected[detected['anomaly']]['anomaly_reason'].value_counts().idxmax()
median_gtn = ph['gross_to_net_pct'].median()
avg_annual_increase = catalog['annual_price_increase_pct'].mean()
highest_wac_drug = catalog.loc[catalog['base_wac'].idxmax(), 'brand_name']
highest_wac = catalog['base_wac'].max()

print('=' * 55)
print('  PHARMA PRICING COPILOT — KEY EDA FINDINGS')
print('=' * 55)
print(f'  Dataset:          {N_DRUGS} drugs × {N_PERIODS} months = {n_total:,} rows')
print(f'  Anomalies found:  {n_anomalies} ({100*n_anomalies/n_total:.1f}%)')
print(f'  Top anomaly type: {top_reason}')
print(f'  Median GTN disc:  {median_gtn*100:.1f}%')
print(f'  Avg annual WAC ↑: {avg_annual_increase*100:.1f}%')
print(f'  Highest WAC drug: {highest_wac_drug} (${highest_wac:,.0f})')
print('=' * 55)